In [0]:
import uuid
from datetime import datetime

GOLD_TABELAS = {
  "dim_calendario": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.gold.dim_calendario AS
      SELECT
        d                          AS data,
        year(d)                    AS ano,
        month(d)                   AS mes,
        date_format(d, 'yyyy-MM')  AS ano_mes,
        quarter(d)                 AS trimestre,
        dayofweek(d)               AS dia_da_semana_num,
        date_format(d, 'EEEE')     AS dia_da_semana,
        weekofyear(d)              AS semana_do_ano,
        (dayofweek(d) IN (1, 7))   AS fim_de_semana
      FROM (
        SELECT explode(sequence(DATE'2016-09-01', DATE'2018-12-31', INTERVAL 1 DAY)) AS d
      )
    """,
    "chave": "data",
    "linhas_esperadas": "SELECT datediff(DATE'2018-12-31', DATE'2016-09-01') + 1",
    "obrigatorias": ["data", "ano_mes"],
    "deve_ser_zero": [
      ("data_de_compra_fora_do_calendario",
       "SELECT COUNT(*) FROM workspace.silver.orders WHERE CAST(order_purchase_ts AS DATE) NOT IN (SELECT data FROM workspace.gold.dim_calendario)"),
    ],
    "reconciliacoes": [],
    "informativos": [],
  },
  "dim_cliente": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.gold.dim_cliente AS
      SELECT
        customer_id,
        customer_unique_id,
        customer_zip_code_prefix AS cep_prefixo,
        customer_city            AS cidade,
        customer_state           AS estado
      FROM workspace.silver.customers
    """,
    "chave": "customer_id",
    "linhas_esperadas": "SELECT COUNT(*) FROM workspace.silver.customers",
    "obrigatorias": ["customer_id", "customer_unique_id", "estado"],
    "deve_ser_zero": [],
    "reconciliacoes": [],
    "informativos": [
      ("clientes_reais", "SELECT COUNT(DISTINCT customer_unique_id) FROM workspace.gold.dim_cliente"),
    ],
  },
  "dim_vendedor": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.gold.dim_vendedor AS
      SELECT
        seller_id,
        seller_zip_code_prefix AS cep_prefixo,
        seller_city            AS cidade,
        seller_state           AS estado,
        flag_cidade_suspeita
      FROM workspace.silver.sellers
    """,
    "chave": "seller_id",
    "linhas_esperadas": "SELECT COUNT(*) FROM workspace.silver.sellers",
    "obrigatorias": ["seller_id", "estado"],
    "deve_ser_zero": [],
    "reconciliacoes": [],
    "informativos": [
      ("cidades_suspeitas", "SELECT COUNT(*) FROM workspace.gold.dim_vendedor WHERE flag_cidade_suspeita"),
    ],
  },
  "dim_produto": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.gold.dim_produto AS
      SELECT
        p.product_id,
        COALESCE(p.product_category_name, 'sem_categoria')                                  AS categoria_pt,
        COALESCE(t.product_category_name_english, p.product_category_name, 'sem_categoria') AS categoria_en,
        CASE
          WHEN p.product_category_name IS NULL THEN 'sem_categoria'
          WHEN t.product_category_name IS NULL THEN 'sem_traducao'
          ELSE 'ok'
        END                                                                                 AS status_categoria,
        p.product_weight_g,
        p.product_length_cm,
        p.product_height_cm,
        p.product_width_cm,
        p.product_photos_qty,
        p.flag_peso_invalido
      FROM workspace.silver.products p
      LEFT JOIN workspace.silver.category_translation t
        ON p.product_category_name = t.product_category_name
    """,
    "chave": "product_id",
    "linhas_esperadas": "SELECT COUNT(*) FROM workspace.silver.products",
    "obrigatorias": ["product_id", "categoria_pt", "categoria_en"],
    "deve_ser_zero": [],
    "reconciliacoes": [],
    "informativos": [
      ("sem_categoria", "SELECT COUNT(*) FROM workspace.gold.dim_produto WHERE status_categoria = 'sem_categoria'"),
      ("sem_traducao",  "SELECT COUNT(*) FROM workspace.gold.dim_produto WHERE status_categoria = 'sem_traducao'"),
    ],
  },
  "dim_cep": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.gold.dim_cep AS
      SELECT
        geolocation_zip_code_prefix        AS cep_prefixo,
        percentile(geolocation_lat, 0.5)   AS lat,
        percentile(geolocation_lng, 0.5)   AS lng,
        COUNT(*)                           AS qtd_pontos
      FROM workspace.silver.geolocation
      WHERE NOT flag_fora_do_brasil
      GROUP BY geolocation_zip_code_prefix
    """,
    "chave": "cep_prefixo",
    "linhas_esperadas": "SELECT COUNT(DISTINCT geolocation_zip_code_prefix) FROM workspace.silver.geolocation WHERE NOT flag_fora_do_brasil",
    "obrigatorias": ["cep_prefixo", "lat", "lng"],
    "deve_ser_zero": [
      ("centro_fora_do_brasil",
       "SELECT COUNT(*) FROM workspace.gold.dim_cep WHERE lat < -34 OR lat > 6 OR lng < -74 OR lng > -34"),
    ],
    "reconciliacoes": [],
    "informativos": [],
  },
    "fact_itens_pedido": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.gold.fact_itens_pedido AS
      SELECT
        i.order_id,
        i.order_item_id,
        i.product_id,
        i.seller_id,
        o.customer_id,
        CAST(o.order_purchase_ts AS DATE)                      AS data_compra,
        o.order_status                                         AS status_pedido,
        i.price                                                AS preco,
        i.freight_value                                        AS frete,
        (o.order_status NOT IN ('canceled', 'unavailable'))    AS conta_na_receita,
        CASE WHEN o.order_status NOT IN ('canceled', 'unavailable')
             THEN i.price
             ELSE CAST(0 AS DECIMAL(12,2))
        END                                                    AS valor_receita,
        2 * 6371.0088 * asin(sqrt(
              pow(sin(radians(cc.lat - cv.lat) / 2), 2) +
              cos(radians(cv.lat)) * cos(radians(cc.lat)) *
              pow(sin(radians(cc.lng - cv.lng) / 2), 2)
        ))                                                     AS distancia_km
      FROM workspace.silver.order_items i
      JOIN workspace.silver.orders o         ON i.order_id    = o.order_id
      LEFT JOIN workspace.gold.dim_cliente  cl ON o.customer_id = cl.customer_id
      LEFT JOIN workspace.gold.dim_vendedor ve ON i.seller_id   = ve.seller_id
      LEFT JOIN workspace.gold.dim_cep      cc ON cl.cep_prefixo = cc.cep_prefixo
      LEFT JOIN workspace.gold.dim_cep      cv ON ve.cep_prefixo = cv.cep_prefixo
    """,
    "chave": ["order_id", "order_item_id"],
    "linhas_esperadas": "SELECT COUNT(*) FROM workspace.silver.order_items",
    "obrigatorias": ["order_id", "order_item_id", "product_id", "seller_id",
                     "customer_id", "data_compra", "preco", "frete", "valor_receita"],
    "deve_ser_zero": [
      ("item_sem_cliente",
       "SELECT COUNT(*) FROM workspace.gold.fact_itens_pedido f LEFT JOIN workspace.gold.dim_cliente d ON f.customer_id = d.customer_id WHERE d.customer_id IS NULL"),
      ("item_sem_vendedor",
       "SELECT COUNT(*) FROM workspace.gold.fact_itens_pedido f LEFT JOIN workspace.gold.dim_vendedor d ON f.seller_id = d.seller_id WHERE d.seller_id IS NULL"),
      ("item_sem_produto",
       "SELECT COUNT(*) FROM workspace.gold.fact_itens_pedido f LEFT JOIN workspace.gold.dim_produto d ON f.product_id = d.product_id WHERE d.product_id IS NULL"),
      ("item_sem_data",
       "SELECT COUNT(*) FROM workspace.gold.fact_itens_pedido f LEFT JOIN workspace.gold.dim_calendario d ON f.data_compra = d.data WHERE d.data IS NULL"),
      ("receita_fora_da_regra",
       "SELECT COUNT(*) FROM workspace.gold.fact_itens_pedido WHERE (conta_na_receita AND valor_receita <> preco) OR (NOT conta_na_receita AND valor_receita <> 0)"),
      ("distancia_negativa",
       "SELECT COUNT(*) FROM workspace.gold.fact_itens_pedido WHERE distancia_km < 0"),
    ],
    "reconciliacoes": [
      ("preco",
       "SELECT SUM(preco) FROM workspace.gold.fact_itens_pedido",
       "SELECT SUM(price) FROM workspace.silver.order_items"),
      ("frete",
       "SELECT SUM(frete) FROM workspace.gold.fact_itens_pedido",
       "SELECT SUM(freight_value) FROM workspace.silver.order_items"),
    ],
    "informativos": [
      ("receita_total",       "SELECT SUM(valor_receita) FROM workspace.gold.fact_itens_pedido"),
      ("itens_fora_receita",  "SELECT COUNT(*) FROM workspace.gold.fact_itens_pedido WHERE NOT conta_na_receita"),
      ("itens_sem_distancia", "SELECT COUNT(*) FROM workspace.gold.fact_itens_pedido WHERE distancia_km IS NULL"),
    ],
  },
  "fact_pedidos": {
    "sql": """
      CREATE OR REPLACE TABLE workspace.gold.fact_pedidos AS
      WITH itens AS (
        SELECT order_id,
               COUNT(*)           AS qtd_itens,
               SUM(price)         AS valor_itens,
               SUM(freight_value) AS frete_total
        FROM workspace.silver.order_items
        GROUP BY order_id
      ),
      pagamentos AS (
        SELECT order_id, SUM(payment_value) AS valor_pago
        FROM workspace.silver.order_payments
        GROUP BY order_id
      ),
      avaliacao AS (
        SELECT order_id, review_score AS nota, qtd_avaliacoes_pedido AS qtd_avaliacoes
        FROM (
          SELECT order_id, review_score, qtd_avaliacoes_pedido,
                 ROW_NUMBER() OVER (
                   PARTITION BY order_id
                   ORDER BY review_creation_ts DESC, review_answer_ts DESC, review_id DESC
                 ) AS rn
          FROM workspace.silver.order_reviews
        )
        WHERE rn = 1
      ),
      distancia AS (
        SELECT order_id, MAX(distancia_km) AS distancia_max_km
        FROM workspace.gold.fact_itens_pedido
        GROUP BY order_id
      ),
      base AS (
        SELECT
          o.order_id,
          o.customer_id,
          CAST(o.order_purchase_ts AS DATE)           AS data_compra,
          o.order_status                              AS status_pedido,
          CAST(o.order_delivered_customer_ts AS DATE) AS data_entrega,
          CAST(o.order_estimated_delivery_ts AS DATE) AS data_prevista,
          COALESCE(i.qtd_itens, 0)                    AS qtd_itens,
          i.valor_itens,
          i.frete_total,
          CASE WHEN o.order_status NOT IN ('canceled', 'unavailable')
               THEN COALESCE(i.valor_itens, CAST(0 AS DECIMAL(12,2)))
               ELSE CAST(0 AS DECIMAL(12,2)) END      AS receita_pedido,
          p.valor_pago,
          a.nota,
          a.qtd_avaliacoes,
          d.distancia_max_km,
          COALESCE(o.order_status = 'delivered'
                   AND o.order_delivered_customer_ts IS NOT NULL, FALSE) AS tem_entrega_valida,
          o.flag_data_entrega_inconsistente
        FROM workspace.silver.orders o
        LEFT JOIN itens      i ON o.order_id = i.order_id
        LEFT JOIN pagamentos p ON o.order_id = p.order_id
        LEFT JOIN avaliacao  a ON o.order_id = a.order_id
        LEFT JOIN distancia  d ON o.order_id = d.order_id
      ),
      calc AS (
        SELECT *,
          CASE WHEN tem_entrega_valida THEN datediff(data_entrega, data_compra)  END AS dias_ate_entrega,
          CASE WHEN tem_entrega_valida THEN datediff(data_entrega, data_prevista) END AS dias_apos_prazo,
          CASE WHEN distancia_max_km IS NULL  THEN NULL
               WHEN distancia_max_km < 100    THEN '1) ate 100 km'
               WHEN distancia_max_km < 300    THEN '2) 100 a 300 km'
               WHEN distancia_max_km < 600    THEN '3) 300 a 600 km'
               WHEN distancia_max_km < 1000   THEN '4) 600 a 1000 km'
               WHEN distancia_max_km < 1500   THEN '5) 1000 a 1500 km'
               ELSE '6) 1500 km ou mais' END                                           AS faixa_distancia
        FROM base
      )
      SELECT *,
        CASE WHEN tem_entrega_valida THEN dias_apos_prazo > 0 END AS entregue_apos_prazo,
        CASE WHEN tem_entrega_valida THEN dias_apos_prazo > 3 END AS entregue_mais_3_dias_apos_prazo
      FROM calc
    """,
    "chave": "order_id",
    "linhas_esperadas": "SELECT COUNT(*) FROM workspace.silver.orders",
    "obrigatorias": ["order_id", "customer_id", "data_compra", "status_pedido", "receita_pedido"],
    "deve_ser_zero": [
      ("pedido_sem_cliente",
       "SELECT COUNT(*) FROM workspace.gold.fact_pedidos f LEFT JOIN workspace.gold.dim_cliente d ON f.customer_id = d.customer_id WHERE d.customer_id IS NULL"),
      ("nota_fora_de_1_a_5",
       "SELECT COUNT(*) FROM workspace.gold.fact_pedidos WHERE nota NOT BETWEEN 1 AND 5"),
      ("atraso_em_pedido_sem_entrega_valida",
       "SELECT COUNT(*) FROM workspace.gold.fact_pedidos WHERE NOT tem_entrega_valida AND entregue_apos_prazo IS NOT NULL"),
      ("mais_3_dias_sem_estar_apos_prazo",
       "SELECT COUNT(*) FROM workspace.gold.fact_pedidos WHERE entregue_mais_3_dias_apos_prazo AND NOT entregue_apos_prazo"),
    ],
    "reconciliacoes": [
      ("receita_pedidos_vs_itens",
       "SELECT SUM(receita_pedido) FROM workspace.gold.fact_pedidos",
       "SELECT SUM(valor_receita) FROM workspace.gold.fact_itens_pedido"),
      ("valor_pago",
       "SELECT SUM(valor_pago) FROM workspace.gold.fact_pedidos",
       "SELECT SUM(payment_value) FROM workspace.silver.order_payments"),
      ("qtd_itens",
       "SELECT SUM(qtd_itens) FROM workspace.gold.fact_pedidos",
       "SELECT COUNT(*) FROM workspace.gold.fact_itens_pedido"),
    ],
    "informativos": [
      ("entregues_validos", "SELECT COUNT(*) FROM workspace.gold.fact_pedidos WHERE tem_entrega_valida"),
      ("apos_prazo",        "SELECT COUNT(*) FROM workspace.gold.fact_pedidos WHERE entregue_apos_prazo"),
      ("mais_3_dias",       "SELECT COUNT(*) FROM workspace.gold.fact_pedidos WHERE entregue_mais_3_dias_apos_prazo"),
      ("pedidos_sem_nota",  "SELECT COUNT(*) FROM workspace.gold.fact_pedidos WHERE nota IS NULL"),
    ],
  },
}

In [0]:
def _um_valor(sql):
    return spark.sql(sql).first()[0]

def validate_gold(nome, spec):
    df = spark.table(f"workspace.gold.{nome}")
    total = df.count()
    esperado = _um_valor(spec["linhas_esperadas"])
    falhas = []
    if total != esperado:
        falhas.append(f"linhas: esperado={esperado} gold={total}")
    cols = spec["chave"] if isinstance(spec["chave"], list) else [spec["chave"]]
    dup = total - df.select(*cols).distinct().count()
    if dup != 0:
        falhas.append(f"chave {cols} duplicada: {dup}")
    for col in spec["obrigatorias"]:
        n = df.filter(f"{col} IS NULL").count()
        if n != 0:
            falhas.append(f"{col} vazia: {n}")
    for rotulo, sql in spec["deve_ser_zero"]:
        n = _um_valor(sql)
        if n != 0:
            falhas.append(f"{rotulo}: {n}")
    for rotulo, sql_a, sql_b in spec["reconciliacoes"]:
        a, b = _um_valor(sql_a), _um_valor(sql_b)
        if a != b:
            falhas.append(f"reconciliacao {rotulo}: {a} != {b}")
    info = {rotulo: _um_valor(sql) for rotulo, sql in spec["informativos"]}
    return esperado, total, falhas, info

In [0]:
def process_gold(run_id=None):
    run_id = run_id or str(uuid.uuid4())
    resultados, auditoria = [], []
    bloqueado = False
    for nome, spec in GOLD_TABELAS.items():
        inicio = datetime.now()
        info, esperado, linhas = {}, None, None
        if bloqueado:
            status, erro = "PULADA", "etapa anterior falhou"
        else:
            try:
                spark.sql(spec["sql"])
                esperado, linhas, falhas, info = validate_gold(nome, spec)
                status = "OK" if not falhas else "DIVERGENTE"
                erro = None if not falhas else "; ".join(falhas)[:200]
            except Exception as e:
                status, erro = "ERRO", str(e)[:200]
            if status != "OK":
                bloqueado = True
        fim = datetime.now()
        resultados.append((nome, esperado, linhas, status, erro, info))
        auditoria.append((run_id, "gold", nome, inicio, fim,
                          (fim - inicio).total_seconds(),
                          status, esperado, linhas, erro))
    record_audit(auditoria)
    return resultados